# Notebook 08: JAX vs PyTorch Mental Model

## Why This Matters

Most practitioners learned deep learning in PyTorch. JAX is not just PyTorch with different syntax -- it has fundamentally different design philosophy. Understanding the conceptual differences helps you avoid bugs (especially around mutable state and randomness), choose the right tool for each project, and translate your PyTorch intuition to JAX. This notebook is a systematic map of the conceptual terrain.


In [ ]:
%pip install -q jax jaxlib numpy torch torchvision

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import torch
import torch.nn as nn
import time
print("JAX:", jax.__version__, "| PyTorch:", torch.__version__)
print("Ready.")

## 1. The Fundamental Difference: Functional vs Stateful

The core philosophical difference:

| Aspect | JAX | PyTorch |
|--------|-----|---------|
| Parameters | Explicit pytrees passed to functions | Implicit, stored inside `nn.Module` |
| Gradients | `jax.grad(f)(x)` returns gradient | `loss.backward()` populates `.grad` |
| Random state | Explicit PRNG keys threaded through | Global state (`torch.manual_seed`) |
| State | Explicit (carry in lax.scan) | Mutable objects |
| Compilation | `@jax.jit` -- must be pure | `torch.compile` -- handles mutation |

**JAX**: functions are pure, state is external, everything is explicit.
**PyTorch**: objects hold state, mutation is expected, convenience is maximized.


In [ ]:
# Gradient computation: JAX vs PyTorch side by side

def loss_fn(x):
    return jnp.sum(x ** 2)

# JAX: functional
x_jax = jnp.array([1.0, 2.0, 3.0])
grad_jax = jax.grad(loss_fn)(x_jax)
print('JAX gradient:', grad_jax)

# PyTorch: imperative with tape
x_pt = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss_pt = torch.sum(x_pt ** 2)
loss_pt.backward()
print('PyTorch gradient:', x_pt.grad)

print('\nKey difference:')
print('  JAX: jax.grad(f) returns a FUNCTION. Calling it gives the gradient.')
print('  PyTorch: loss.backward() MUTATES the .grad attribute of leaf tensors.')
print('  JAX gradient has no side effects.')
print('  PyTorch gradient accumulates (requires zero_grad() before each step).')

## 2. Module Parameters: Explicit vs Implicit

In PyTorch, an `nn.Module` stores parameters internally. When you call `model(x)`, it reads its own `weight` and `bias` attributes.

In JAX with Flax NNX, parameters are attributes of the module but are managed through `nnx.state()`. The critical difference: JAX's transforms (grad, jit, vmap) need to work on **pure functions**, so Flax separates the **structure** (how to compute) from the **state** (the actual weight values).


In [ ]:
# PyTorch: parameters inside module
class TorchLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.linear = nn.Linear(in_f, out_f)
    
    def forward(self, x):
        return self.linear(x)  # reads self.linear.weight implicitly

pt_model = TorchLinear(4, 8)
x_pt = torch.randn(2, 4)
y_pt = pt_model(x_pt)
print('PyTorch output shape:', y_pt.shape)
print('PyTorch param count:', sum(p.numel() for p in pt_model.parameters()))

# JAX/Flax NNX: parameters as attributes
from flax import nnx
flax_model = nnx.Linear(4, 8, rngs=nnx.Rngs(0))
x_jax = jnp.ones((2, 4))
y_jax = flax_model(x_jax)
print('\nFlax NNX output shape:', y_jax.shape)

state = nnx.state(flax_model)
n_params = sum(v.value.size for v in jax.tree_util.tree_leaves(state))
print('Flax NNX param count:', n_params)

print('\nKey difference: In Flax NNX, nnx.state(model) returns an explicit pytree.')
print('This pytree is what jax.grad differentiates -- explicit state, not side effects.')

## 3. Random State: Explicit vs Global

The single most surprising difference for PyTorch users: JAX has no global random state.

**PyTorch** global seed:
```python
torch.manual_seed(42)
x = torch.randn(3)  # uses and advances global state
y = torch.randn(3)  # different result, global state advanced
```

**JAX** explicit keys:
```python
key = jax.random.PRNGKey(42)
key, subkey = jax.random.split(key)
x = jax.random.normal(subkey, (3,))  # always same result for this key
```

The JAX approach is better for reproducibility: the result of any JAX program depends only on its explicit inputs, with no hidden global state.


In [ ]:
# Random state comparison

print('=== PyTorch: global state ===')
torch.manual_seed(42)
x1 = torch.randn(3)
x2 = torch.randn(3)
print('x1:', x1)
print('x2:', x2)  # different -- global state advanced

# Reset and repeat
torch.manual_seed(42)
x1_repeat = torch.randn(3)
print('x1 repeated:', x1_repeat, '(same as x1)')

print('\n=== JAX: explicit keys ===')
key = jax.random.PRNGKey(42)
key, k1 = jax.random.split(key)
key, k2 = jax.random.split(key)
x1_jax = jax.random.normal(k1, (3,))
x2_jax = jax.random.normal(k2, (3,))
print('x1:', x1_jax)
print('x2:', x2_jax)  # different because different keys

# Reproduce any result by reusing the same key
x1_repeat_jax = jax.random.normal(k1, (3,))
print('x1 repeated:', x1_repeat_jax, '(exactly same as x1)')
print('Same key -> same result, always.')

## 4. JIT Compilation: @jax.jit vs torch.compile

| Feature | `@jax.jit` | `torch.compile` |
|---------|-----------|-----------------|
| Requires pure functions | Yes | No (handles mutation) |
| Dynamic shapes | Recompiles per shape | Supports via guards |
| Debug difficulty | High (Python control flow hidden) | Lower (fallback to eager) |
| Compilation backend | XLA | Triton / inductor |
| Available since | JAX 0.1 (2018) | PyTorch 2.0 (2023) |

**JAX JIT philosophy**: you promise the compiler your function is pure. In exchange, it can fully optimize it.
**torch.compile**: captures and compiles, but handles the messy reality of mutable Python objects.


In [ ]:
# JIT comparison

# JAX jit
@jax.jit
def jax_fn(x, W):
    return jax.nn.gelu(x @ W)

key = jax.random.PRNGKey(0)
x_jax = jax.random.normal(key, (128, 256))
W_jax = jax.random.normal(key, (256, 128))

_ = jax_fn(x_jax, W_jax).block_until_ready()  # compile
t0 = time.perf_counter()
for _ in range(500):
    _ = jax_fn(x_jax, W_jax).block_until_ready()
jax_time = (time.perf_counter() - t0) / 500 * 1000

# PyTorch eager
x_pt = torch.randn(128, 256)
W_pt = torch.randn(256, 128)

def pt_fn(x, W):
    return torch.nn.functional.gelu(x @ W)

t0 = time.perf_counter()
for _ in range(500):
    _ = pt_fn(x_pt, W_pt)
pt_eager_time = (time.perf_counter() - t0) / 500 * 1000

# PyTorch compiled
try:
    pt_fn_compiled = torch.compile(pt_fn)
    _ = pt_fn_compiled(x_pt, W_pt)  # compile
    t0 = time.perf_counter()
    for _ in range(500):
        _ = pt_fn_compiled(x_pt, W_pt)
    pt_compiled_time = (time.perf_counter() - t0) / 500 * 1000
    print(f'PyTorch compiled: {pt_compiled_time:.4f} ms')
except Exception as e:
    print(f'torch.compile not available: {e}')
    pt_compiled_time = pt_eager_time

print(f'JAX jit:         {jax_time:.4f} ms')
print(f'PyTorch eager:   {pt_eager_time:.4f} ms')

## 5. Translation Table: PyTorch to JAX

| PyTorch concept | JAX/Flax equivalent |
|----------------|---------------------|
| `torch.Tensor` | `jax.Array` (immutable) |
| `tensor.backward()` | `jax.grad(f)(x)` |
| `nn.Module` | `nnx.Module` |
| `optimizer.step()` | `optax.update(grads, state)` |
| `torch.manual_seed(n)` | `jax.random.PRNGKey(n)` |
| `torch.randn(...)` | `jax.random.normal(key, ...)` |
| `@torch.jit.script` | `@jax.jit` |
| `torch.compile` | `@jax.jit` |
| `model.train()` | `model(x, training=True)` |
| `model.eval()` | `model(x, training=False)` |
| `loss.backward()` | Implicit in `jax.grad` |
| `optimizer.zero_grad()` | Not needed (no gradient accumulation) |
| `torch.no_grad()` | `jax.lax.stop_gradient(x)` |
| `for i in range(T): ...` | `lax.scan(fn, init, xs)` |


In [ ]:
# Side-by-side training step comparison

print('=== PyTorch training step ===')
pt_code = """
# PyTorch
model = nn.Linear(4, 2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for x_batch, y_batch in dataloader:
    optimizer.zero_grad()           # clear previous gradients
    logits = model(x_batch)         # forward (reads model.weight implicitly)
    loss = criterion(logits, y_batch)
    loss.backward()                 # populate .grad on all params
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()                # update params from .grad
"""
print(pt_code)

print('=== JAX/Flax training step ===')
jax_code = """
# JAX + Flax NNX + Optax
model = nnx.Linear(4, 2, rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(1e-3)
))

@nnx.jit
def train_step(model, optimizer, x, y):
    def loss_fn(model):               # closure captures x, y
        logits = model(x)
        return cross_entropy(logits, y)
    loss, grads = nnx.value_and_grad(loss_fn)(model)  # explicit grad
    optimizer.update(grads)           # update (no .backward(), no zero_grad)
    return loss

for x_batch, y_batch in dataloader:
    loss = train_step(model, optimizer, x_batch, y_batch)
"""
print(jax_code)

## 6. When to Choose JAX vs PyTorch

**Choose JAX when**:
- Training on TPUs (JAX is Google's first-class citizen; PyTorch TPU support is less mature)
- You need per-sample gradients frequently (vmap makes this trivial)
- Research requiring custom autodiff (custom_vjp, implicit differentiation)
- You want composable transforms (meta-learning, NTK, HVP)
- The codebase is entirely in JAX (Flax, Optax, Orbax, Grain ecosystem)

**Choose PyTorch when**:
- Production deployment (TorchServe, ONNX, TensorRT ecosystem)
- Dynamic shapes are common (variable-length sequences without padding)
- Team is more familiar with PyTorch
- Using PyTorch-native libraries (Detectron2, HuggingFace Transformers, etc.)
- Debugging is critical (PyTorch's eager mode and debuggability are superior)


## Summary

### Key Concepts

| Dimension | JAX | PyTorch |
|-----------|-----|---------|
| Philosophy | Functional, explicit state | Object-oriented, mutable state |
| Gradients | `jax.grad(f)(x)` -- functional | `loss.backward()` -- side effect |
| Randomness | Explicit PRNG keys | Global seed |
| JIT | Must be pure | Handles mutation |
| Best for | TPU, research transforms, DP | Production, dynamic shapes |

**Next**: `09_advanced_transforms.ipynb` -- Jacobians, implicit differentiation, and meta-learning.
